<a href="https://colab.research.google.com/github/elias9080dm/XenoTox/blob/main/QSAR_Classification_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Observaciones***

Nueva versión con modificaciones:\
**Funcional completamente para modelos individuales y stacking**

Nota: Shap no se aplica para stacking (es muy tardado)







# **QSAR Clasificación Binaria**

Inspirado en el artículo de Kotli et al. (2025), este cuaderno permite construir modelos QSAR binarios sobre datos desbalanceados.

**Incluye:**
- Estandarización química (Greg Landrum, 2021)
- Modelos: Random Forest, XGBoost, SVM, Logistic Regression
- Métrica principal: MCC
- Selección de variables con GA
- Interpretabilidad con SHAP
- Y-scrambling
- Dominio de aplicabilidad (Leverage)
- Guardado automático de modelos, gráficas y métricas

## **1. Imports**

In [1]:
# Instalación de librerías necesarias
%pip install pandas numpy scikit-learn scikit-optimize xgboost rdkit shap optuna deap


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 10.7 MB/s eta 0:00:00


In [2]:
# General
import joblib, io
from joblib import Parallel, delayed
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from tqdm import tqdm
import warnings, contextlib, base64
from psutil import cpu_count
from IPython.display import HTML

# Curación y rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, AllChem, Draw
from rdkit.Chem.MolStandardize import rdMolStandardize

# Selección de variables (GA)
from deap import base, creator, tools, algorithms
import random

# Machine Learning
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Balanceo y pipeline general
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer

# Hiperoptimización
import optuna
from skopt.space import Real as SkReal, Integer as SkInteger, Categorical
from sklearn.base import clone, BaseEstimator, TransformerMixin
from collections import Counter

# Métricas y análisis SHAP
from sklearn.metrics import (
    make_scorer, accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, confusion_matrix, RocCurveDisplay,
    auc, roc_curve, PrecisionRecallDisplay, precision_recall_curve
    )
import shap

## **2. Configuración**

In [3]:
confidence = 'lc'   #'hc', 'lc'
target = 'ahr' # 'ahr', 'pxr', 'car' Los nombres de los archivos deben ser {target}_ligands.csv
model_name = 'lr'  # 'rf', 'xgb', 'svm', 'lr'
use_stacking = False
use_hyperopt = True  # Método optuna
n_jobs = -1  # Utilizar todos los CPU disponibles
random_state = 42

In [4]:
from google.colab import drive
import sys, os
from pathlib import Path

# Montar Google Drive
drive.mount('/content/drive')

# Definir directorio base del proyecto
BASE_DIR = Path("/content/drive/MyDrive/QSAR/xenotox")

# Crear estructura de carpetas para outputs
os.makedirs(f"{BASE_DIR}/outputs_{confidence}/{target}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs_{confidence}/{target}/plots", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs_{confidence}/{target}/reports", exist_ok=True)

Mounted at /content/drive


## **3. Curación de base de datos**

In [ ]:
# Apagar logs de RDKit
RDLogger.DisableLog("rdApp.*")

def curate_csv_data(
    csv_path,
    smiles_col="SMILES",
    activity_col="Agonist_Activity"
):
    # =========================================================================
    # LOAD
    # =========================================================================
    df_raw = pd.read_csv(csv_path)
    initial_count = len(df_raw)

    # =========================================================================
    # STEP 1 – BASIC SANITY (NaNs)
    # =========================================================================
    df = df_raw.dropna(subset=[smiles_col, activity_col]).copy()

    # =========================================================================
    # STEP 2 – SMILES PARSING
    # =========================================================================
    df["mol"] = df[smiles_col].map(Chem.MolFromSmiles)
    df = df[df["mol"].notna()].copy()

    # =========================================================================
    # STEP 3 – ORGANIC FILTER
    # =========================================================================
    def is_organic(mol):
        return any(a.GetAtomicNum() == 6 for a in mol.GetAtoms())

    df = df[df["mol"].map(is_organic)].copy()

    # =========================================================================
    # STEP 4 – STANDARDIZATION
    # =========================================================================
    normalizer = rdMolStandardize.Normalize
    uncharger = rdMolStandardize.Uncharger()
    tautomer_enum = rdMolStandardize.TautomerEnumerator()

    def standardize(mol):
        try:
            # 4.1 Normalize (functional group normalization)
            mol = normalizer(mol)

            # 4.2 Fragment parent (salt stripping)
            mol = rdMolStandardize.FragmentParent(mol)

            # 4.3 Uncharge (QSAR-consistent)
            mol = uncharger.uncharge(mol)

            # 4.4 Canonical tautomer
            mol = tautomer_enum.Canonicalize(mol)

            return mol
        except Exception:
            return None

    df["mol_std"] = df["mol"].map(standardize)
    df = df[df["mol_std"].notna()].copy()

    # =========================================================================
    # STEP 5 – CANONICAL SMILES FROM STANDARDIZED MOL
    # =========================================================================
    df["SMILES_std"] = df["mol_std"].map(
        lambda m: Chem.MolToSmiles(m, canonical=True)
    )

    # Eliminar wildcards residuales
    df = df[~df["SMILES_std"].str.contains(r"\*", na=False)].copy()

    # =========================================================================
    # STEP 6 – DEDUPLICATION (Priorizando 'active' si hay duplicados de SMILES)
    # =========================================================================
    # Mapear 'active' a un valor mayor para priorizarlo
    activity_mapping = {'active': 1, 'inactive': 0}
    df['activity_rank'] = df[activity_col].map(activity_mapping)

    # Ordenar para que 'active' aparezca primero en caso de SMILES_std duplicados
    df = df.sort_values(by=['SMILES_std', 'activity_rank'], ascending=[True, False]).copy()

    # Eliminar duplicados, manteniendo la primera (ahora priorizada 'active')
    df = df.drop_duplicates(subset="SMILES_std", keep="first").copy()

    # Eliminar la columna temporal 'activity_rank'
    df = df.drop(columns=['activity_rank']).copy()

    # =========================================================================
    # FINALIZATION
    # =========================================================================
    df_final = (
        df[[activity_col, "SMILES_std"]]
        .rename(columns={"SMILES_std": "SMILES"})
        .reset_index(drop=True)
    )

    final_count = len(df_final)
    print(
        f"Curación completa: "
        f"{final_count} moléculas válidas "
        f"(de {initial_count} iniciales)."
    )

    return df_final

# Usar la función de curación en el archivo CSV correspondiente
raw_path = f"{BASE_DIR}/ligands/{target}/{target}_ligands_{confidence}.csv"
curated_path = f"{BASE_DIR}/outputs_{confidence}/{target}/reports/{target}_ligands_curated.csv"

df_curated = curate_csv_data(raw_path)
df_curated.to_csv(curated_path, index=False)

display(df_curated.head())

## **4. Cálculo de descriptores**

In [ ]:
# ==========================================================
# 1. Función de cálculo
# ==========================================================

def calcular_descriptores(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return [np.nan] * len(Descriptors._descList)

    results = []

    for _, func in Descriptors._descList:
        try:
            d = func(mol)
            if d is None or isinstance(d, complex):
                d = np.nan
        except Exception:
            d = np.nan

        results.append(d)

    return results


# ==========================================================
# 2. Configuración de hilos
# ==========================================================

n_cores = cpu_count()
print(f"Iniciando cálculo en paralelo con {n_cores} núcleos...")


# ==========================================================
# 3. Ejecución paralela
# ==========================================================

smiles_list = df_curated["SMILES"].tolist()

resultados = Parallel(n_jobs=n_cores)(
    delayed(calcular_descriptores)(s)
    for s in tqdm(smiles_list, desc="Progreso RDKit")
)


# ==========================================================
# 4. Crear DataFrame completo de descriptores
# ==========================================================

feature_names = [name for name, _ in Descriptors._descList]

X_full = pd.DataFrame(
    resultados,
    index=df_curated.index,
    columns=feature_names
)


# ==========================================================
# 5. Eliminar moléculas con cualquier NaN en descriptores
# ==========================================================

mask_valid = ~X_full.isna().any(axis=1)

X = X_full.loc[mask_valid].copy()
df_curated = df_curated.loc[mask_valid].copy()


# ==========================================================
# 6. Resetear índices de forma consistente
# ==========================================================

X.reset_index(drop=True, inplace=True)
df_curated.reset_index(drop=True, inplace=True)

y = df_curated["Agonist_Activity"]  # <-- SIN .values


# ==========================================================
# 7. Reporte final
# ==========================================================

print(f"\nProceso completado.")
print(f"Dataset: {target} | Moléculas finales: {len(X)} | Descriptores: {X.shape[1]}")

out_path = f"{BASE_DIR}/outputs_{confidence}/{target}/reports/{target}_ligands_descriptors.csv"
df_curated.to_csv(out_path, index=False)

print(f"Archivo guardado en: {out_path}")

descriptor_names_full = feature_names.copy()

### 4.1 Visualizar distribución de clases

In [ ]:
# Histograma de clases en la base de datos curada
activity_col = "Agonist_Activity"
plt.figure(figsize=(4, 4))
ax = sns.countplot(x=activity_col, data=df_curated, edgecolor='black', color='chocolate')
plt.title(f"Distribución de clases ({target})")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")

for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{int(height)}",
        (p.get_x() + p.get_width() / 2., height),
        ha='center',
        va='bottom',
        xytext=(0, -1),
        textcoords='offset points'
    )

plt.tight_layout()

# Guardar figura
out_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/{target}_class_distribution.png"
plt.savefig(out_path, dpi=300)
plt.show()

print(f"Histograma guardado en: {out_path}")


## **5. División estratificada y codificación**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    test_size=0.3,
    random_state=random_state
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# ==========================================================
#       CODIFICACIÓN MANUAL DE ETIQUETAS
# ==========================================================

# Train
y_train_enc = y_train.map({"inactive": 0, "active": 1})

# Test
y_test_enc = y_test.map({"inactive": 0, "active": 1})


print("Distribución train:", np.bincount(y_train_enc))
print("Distribución test :", np.bincount(y_test_enc))

print("\nConvención:")
print("  1 = active (clase positiva)")
print("  0 = inactive")

## **Funciones auxiliares**

In [ ]:
# == Preprocesador ===
def build_preprocessor():

    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

# === Función segura de StratifiedKFold ===
def safe_stratified_kfold(y, max_folds=5, shuffle=True, random_state=42):
    counts = Counter(y)
    min_class_count = min(counts.values())
    n_splits = min(max_folds, min_class_count)
    if n_splits < 2:
        raise ValueError(f"No se puede usar validación cruzada: solo hay {min_class_count} muestras en la clase menor.")
    return StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

## **Filtrado de descriptores**

In [ ]:
from sklearn.feature_selection import VarianceThreshold
import pandas as pd

print("Aplicando VarianceThreshold...")

var_filter = VarianceThreshold(threshold=0.01)
X_train_var_array = var_filter.fit_transform(X_train)

# Recuperar nombres retenidos
var_mask = var_filter.get_support()
var_features = X_train.columns[var_mask]

# Reconstruir DataFrame con índices originales
X_train_var = pd.DataFrame(
    X_train_var_array,
    columns=var_features,
    index=X_train.index
)

# Aplicar mismas columnas a test
X_test_var = X_test[var_features]
print("Features eliminadas por varianza", len(X_train.columns) - len(var_features))
print("Features después de varianza:", len(var_features))

In [ ]:
print("Aplicando filtro de alta correlación...")

corr_matrix = pd.DataFrame(X_train_var, columns=var_features).corr().abs()

# Matriz triangular superior
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Columnas a eliminar
corr_threshold = 0.9
to_drop_corr = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > corr_threshold)
]

# Features finales tras filtro estructural
filtered_features = [f for f in var_features if f not in to_drop_corr]

# Aplicar a train y test
X_train_filtered = pd.DataFrame(X_train_var, columns=var_features)[filtered_features]
X_test_filtered = X_test_var[filtered_features]

print("Features eliminadas por correlación:", len(to_drop_corr))
print("Features finales post-filtrado:", len(filtered_features))

## **Selección de variables (GA)**

In [ ]:
cv = safe_stratified_kfold( y_train_enc, max_folds=5, random_state=random_state )

def evaluate_individual(individual, X, y):
    n_selected = sum(individual)
    MIN_FEATURES = max(5, int(0.01 * len(individual)))

    # Penalización
    if n_selected < MIN_FEATURES:
        return -1.0,

    selected_indices = [i for i, bit in enumerate(individual) if bit == 1]
    X_selected = X[:, selected_indices]

    try:
        model = LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                class_weight="balanced",
                max_iter=5000
            )

        mcc_scorer = make_scorer(matthews_corrcoef)

        scores = cross_val_score(model, X_selected, y, cv=cv, scoring=mcc_scorer)

        # Penalización por complejidad
        alpha = 0.5
        fitness = scores.mean() - alpha * (n_selected / len(individual))
        return fitness,

    except Exception as e:
      print("Error en evaluate_individual:", repr(e))
      raise

n_features = X_train_filtered.shape[1]

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=n_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate_individual, X=X_train_filtered.values, y=y_train_enc)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.005)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
np.random.seed(42)
random.seed(42)
pop = toolbox.population(n=50)
hof = tools.HallOfFame(1)

stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("avg", np.mean)
stats.register("max", np.max)
stats.register("min", np.min)

pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.3,
                                ngen=30, stats=stats, halloffame=hof, verbose=True)

In [ ]:
best_ind = hof[0]

selected_features = [
    descriptor_names_full[i]
    for i, bit in enumerate(best_ind)
    if bit == 1
]

# Imprimir resumen
print(f"\n Número de descriptores seleccionados: {len(selected_features)}")
print(" Descriptores:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:2d}. {f}")

In [ ]:
print(len(descriptor_names_full))
print(X_train_filtered.shape[1])
print(len(selected_features))

## **7. Hiperoptimización Optuna**

In [ ]:
# ==================================================
# MAIN OPTIMIZATION FUNCTION
# ==================================================
def optimize_model(X_train_proc, y_encoded, model_name, n_jobs=1, random_state=42):
    """
    Optimiza un modelo base usando Optuna.
    El preprocesamiento de X debe realizarse externamente
    (pipeline entrenado SOLO con train).
    """

    # ==================================================
    # 1. Métrica
    # ==================================================
    mcc_scorer = make_scorer(matthews_corrcoef)

    # ==================================================
    # 2. CV estratificado seguro
    # ==================================================
    cv = safe_stratified_kfold(
        y_encoded,
        max_folds=5,
        random_state=random_state
    )

    # ==================================================
    # 3. Balanceo para XGBoost (SIN SMOTE)
    # ==================================================
    scale_pos_weight = None
    if model_name == "xgb":
        counter = Counter(y_encoded)
        scale_pos_weight = counter[0] / counter[1]

    # ==================================================
    # 4. Modelo base (SIN preprocessing)
    # ==================================================
    if model_name == "xgb":
        base_model = XGBClassifier(
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=n_jobs,
            tree_method="hist",
            scale_pos_weight=scale_pos_weight
        )

    elif model_name == "rf":
        base_model = RandomForestClassifier(
            n_jobs=n_jobs,
            class_weight="balanced",
            random_state=random_state
        )

    elif model_name == "svm":
        base_model = SVC(
            probability=True,
            class_weight="balanced",
            kernel="rbf",
            random_state=random_state
        )
    elif model_name == "lr":
        base_model = LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            class_weight="balanced",
            max_iter=2000,
            random_state=random_state,
            n_jobs=n_jobs
        )

    else:
        raise ValueError(f"Modelo no soportado: {model_name}")

    # ==================================================
    # 5. Función objetivo (OPTUNA REAL)
    # ==================================================
    def objective(trial):
        current_model = clone(base_model)

        if model_name == "xgb":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 800),
                "max_depth": trial.suggest_int("max_depth", 3, 5),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 0.9),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
                "gamma": trial.suggest_float("gamma", 0.0, 0.3),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 1.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            }

        elif model_name == "rf":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 300, 800),
                "max_depth": trial.suggest_int("max_depth", 6, 12),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
                "max_features": trial.suggest_categorical(
                    "max_features", ["sqrt", "log2"]
                ),
            }

        elif model_name == "svm":
            params = {
                "C": trial.suggest_float("C", 0.1, 50, log=True),
                "gamma": trial.suggest_float("gamma", 1e-4, 0.1, log=True),
            }

        elif model_name == "lr":
            params = {
                "C": trial.suggest_float("C", 1e-2, 100, log=True)
            }

        current_model.set_params(**params)

        scores = cross_val_score(
            current_model,
            X_train_proc,
            y_encoded,
            scoring=mcc_scorer,
            cv=cv,
            n_jobs=1
        )

        return scores.mean()

    # ==================================================
    # 6. Optuna study
    # ==================================================
    study = optuna.create_study(
        study_name=f"{model_name}_study",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
    )

    study.optimize(objective, n_trials=25)

    # ==================================================
    # 7. Entrenamiento final con MEJORES parámetros
    # ==================================================
    final_model = clone(base_model)
    final_model.set_params(**study.best_params)
    final_model.fit(X_train_proc, y_encoded)

    # Guardar metadata útil
    final_model.best_params_ = study.best_params
    final_model.best_score_ = study.best_value

    return final_model


In [ ]:
def train_stacking_model(
    X_train_proc,
    y_train_enc,
    n_jobs=1,
    random_state=42
):
    """
    Entrena un StackingClassifier sobre datos YA preprocesados.

    Parámetros
    ----------
    X_train_proc : np.ndarray
        Matriz de entrenamiento preprocesada (clean + impute + variance + scale)
    y_train_enc : np.ndarray
        Etiquetas codificadas
    """

    print("Entrenando StackingClassifier")

    # ==================================================
    # 1. Modelos base (reciben X YA preprocesado)
    # ==================================================
    base_model_names = ["rf", "xgb", "svm"]
    estimators = []

    print("Entrenando modelos base para stacking:")

    for name in base_model_names:
        print(f"  → Optimizando {name.upper()} con Optuna")
        model = optimize_model(
            X_train_proc,
            y_train_enc,
            model_name=name
        )
        estimators.append((name, model))

    # ==================================================
    # 2. CV del stacking
    # ==================================================
    try:
        cv_stack = safe_stratified_kfold(
            y_train_enc,
            max_folds=5,
            random_state=random_state
        )
    except ValueError:
        cv_stack = StratifiedKFold(
            n_splits=2,
            shuffle=True,
            random_state=random_state
        )

    # ==================================================
    # 3. Meta-modelo
    # ==================================================
    print("Entrenando Meta-modelo...")
    meta_model = LogisticRegression(
        penalty="l2",
        C=0.1,
        class_weight="balanced",
        max_iter=1000,
        solver="liblinear",
        random_state=random_state
    )

    # ==================================================
    # 4. Stacking final
    # ==================================================
    print("Armando StackingClassifier...")
    stacking_model = StackingClassifier(
        estimators=estimators,
        final_estimator=meta_model,
        stack_method="predict_proba",
        cv=cv_stack,
        n_jobs=n_jobs,
        passthrough=False
    )

    stacking_model.fit(X_train_proc, y_train_enc)

    return stacking_model


## **8. Entrenamiento y stacking**

In [ ]:
# ==========================================================
#       SILENCIAR WARNINGS
# ==========================================================

warnings.filterwarnings("ignore", category=UserWarning, module="multiprocessing")
warnings.filterwarnings("ignore", category=UserWarning, module="pkg_resources")

# ==========================================================
#       OCUPAR MATRIZ DE DESCRIPTORES SELECCIONADOS POR GA
# ==========================================================

X_train_ft = X_train[selected_features]
X_test_ft  = X_test[selected_features]

# ==========================================================
#       PREPROCESADOR GLOBAL (ÚNICO)
# ==========================================================

trained_preprocessor = build_preprocessor()
X_train_preprocessed = trained_preprocessor.fit_transform(X_train_ft)
X_test_preprocessed  = trained_preprocessor.transform(X_test_ft)

print("Preprocesador global entrenado")

# ==========================================================
#       ENTRENAMIENTO DEL MODELO
# ==========================================================

if use_stacking:
    final_model = train_stacking_model(
        X_train_preprocessed,
        y_train_enc
    )
    model_type = "stacking"
    model_filename = f"{BASE_DIR}/outputs_{confidence}/{target}/models/best_model_{target}_{model_type}_optuna.pkl"
else:
    final_model = optimize_model(
        X_train_preprocessed,
        y_train_enc,
        model_name=model_name
    )
    model_type = model_name
    model_filename = f"{BASE_DIR}/outputs_{confidence}/{target}/models/best_model_{target}_{model_name}_optuna.pkl"

# ==========================================================
#       GUARDADO UNIFICADO Y CONSISTENTE
# ==========================================================

model_components = {
    "model": final_model,
    "descriptor_names_full": descriptor_names_full,
    "selected_features": selected_features,
    "trained_preprocessor": trained_preprocessor,
    "target": target,
    "model_type": model_type,
    "class_mapping": {0: "inactive", 1: "active"},
    "positive_class": 1
}

joblib.dump(model_components, model_filename)
print(f"\nModelo y componentes guardados en: {model_filename}")

In [ ]:
# Opcional: Cargar modelo ya entrenado

# Asegurarse de que model_type esté definido (esto se define en la celda 8. Entrenamiento y stacking)
if use_stacking:
    model_filename = f"{BASE_DIR}/outputs_{confidence}/{target}/models/best_model_{target}_stacking_optuna.pkl"
    model_components = joblib.load(model_filename)

    # Componentes
    final_model = model_components["model"]
    selected_features = model_components["selected_features"]
    trained_preprocessor = model_components["trained_preprocessor"]
    target = model_components["target"]
    model_type = model_components["model_type"]
    class_mapping = model_components["class_mapping"]
    positive_class = model_components["positive_class"]


    print(f"Modelo cargado desde: {model_filename}")
else:
    model_type = model_name

    model_filename = f"{BASE_DIR}/outputs_{confidence}/{target}/models/best_model_{target}_{model_type}_optuna.pkl"
    model_components = joblib.load(model_filename)

    final_model = model_components["model"]
    selected_features = model_components["selected_features"]
    trained_preprocessor = model_components["trained_preprocessor"]
    target = model_components["target"]
    model_type = model_components["model_type"]
    class_mapping = model_components["class_mapping"]
    positive_class = model_components["positive_class"]

    print(f"Modelo cargado desde: {model_filename}")

# Preprocesamiento
X_train_ft = X_train[selected_features]
X_test_ft  = X_test[selected_features]

X_train_preprocessed = trained_preprocessor.transform(X_train_ft)
X_test_preprocessed  = trained_preprocessor.transform(X_test_ft)

y_test_enc = y_test.map({"inactive": 0, "active": 1})

In [ ]:
print(final_model)

## **9. Validación externa**

In [ ]:
def compute_metrics(final_model, X_test, y_test, model_name, target, model_type):
    """
    X_test debe estar preprocesado previamente.
    y_test debe estar codificado como {0,1}
    Threshold fijo = 0.5 (coherente con entrenamiento)
    """

    # ==========================================================
    # 1. PROBABILIDADES
    # ==========================================================
    classes = list(final_model.classes_)
    if 1 not in classes:
        raise ValueError("La clase positiva (1) no está en final_model.classes_")

    pos_index = classes.index(1)
    y_proba = final_model.predict_proba(X_test)[:, pos_index]

    # Threshold fijo coherente con entrenamiento
    y_pred = (y_proba >= 0.5).astype(int)

    # ==========================================================
    # 2. MÉTRICAS
    # ==========================================================
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # ROC AUC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    # PR AUC
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(recall_curve, precision_curve)

    # ==========================================================
    # 3. GUARDAR MÉTRICAS
    # ==========================================================
    metrics_df = pd.DataFrame([{
        "Model": model_name,
        "Target": target,
        "Threshold": 0.5,
        "Accuracy": acc,
        "Bal_Accuracy": bal_acc,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1_score": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "MCC": mcc
    }])

    if model_type == "stacking":
        metrics_path = f"{BASE_DIR}/outputs_{confidence}/{target}/reports/metrics_{target}_stacking.csv"
    else:
        metrics_path = f"{BASE_DIR}/outputs_{confidence}/{target}/reports/metrics_{target}_{model_name}.csv"

    metrics_df.to_csv(metrics_path, index=False)
    print(f"Métricas guardadas en: {metrics_path}")
    display(metrics_df)

    # ==========================================================
    # 4. CURVAS (USANDO MISMAS PROBABILIDADES)
    # ==========================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    RocCurveDisplay.from_predictions(
        y_test,
        y_proba,
        ax=ax1,
        name=f"{model_name}"
    )
    ax1.set_title(f"ROC Curve ({target} - {model_name})")
    ax1.grid(True, alpha=0.3)

    PrecisionRecallDisplay.from_predictions(
        y_test,
        y_proba,
        ax=ax2,
        name=f"{model_name}"
    )
    ax2.set_title(f"PR Curve ({target} - {model_name})")
    ax2.grid(True, alpha=0.3)

    if model_type == "stacking":
        plot_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/roc_pr_{target}_stacking.png"
    else:
        plot_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/roc_pr_{target}_{model_name}.png"

    plt.tight_layout()
    plt.savefig(plot_path, dpi=300)
    plt.show()

    print(f"Gráficas guardadas en: {plot_path}")

    return {
        "metrics_df": metrics_df,
        "confusion_matrix": cm,
        "y_pred": y_pred,
        "y_proba": y_proba,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

In [ ]:
# ===============================
# Evaluación
# ===============================
results = compute_metrics(
    final_model=final_model,
    X_test=X_test_preprocessed,
    y_test=y_test_enc,
    model_name=model_name,
    target=target,
    model_type=model_type
)

print("\nEvaluación completada.")

## **10. Interpretabilidad SHAP**

### **Nota:**
El análisis SHAP se realiza únicamente sobre modelos base (de stacking) y modelos individuales.
En el caso de stacking, la interpretabilidad química se evalúa a nivel de modelos base (nivel 0), no sobre el StackingClassifier completo.

In [ ]:
def process_shap_top20(
    estimator,
    model_name,
    X_proc,
    X_background_proc,
    feature_names,
    target,
    positive_class=1,
    max_samples=100,
    top_n=20,
    random_state=42
):

    # ==========================================================
    # 1. Omitir stacking (meta-modelo no usa descriptores)
    # ==========================================================
    if model_name.lower() == "stacking":
        print(
            f"SHAP omitido para {model_name}: "
            "el meta-modelo opera sobre salidas de modelos base."
        )
        return

    print(f"--- SHAP Top-{top_n}: {model_name} ---")

    rng = np.random.default_rng(random_state)

    # ==========================================================
    # 2. Submuestreo estable
    # ==========================================================
    n_samples = min(max_samples, X_proc.shape[0])
    idx = rng.choice(X_proc.shape[0], size=n_samples, replace=False)
    X_shap = X_proc[idx]

    model_cls = estimator.__class__.__name__.lower()

    # ==========================================================
    # 3. TREE MODELS
    # ==========================================================
    if "xgb" in model_cls or "randomforest" in model_cls:

        explainer = shap.TreeExplainer(
            estimator,
            data=X_background_proc, # Added background data
            model_output="probability",
            feature_perturbation="interventional" # Explicitly set to 'interventional'
        )

        shap_values = explainer(X_shap)

        # Extraer clase positiva si es binario
        if shap_values.values.ndim == 3:
            class_index = list(estimator.classes_).index(positive_class)
            shap_vals = shap_values.values[:, :, class_index]
        else:
            shap_vals = shap_values.values

    # ==========================================================
    # 4. SVM (Kernel SHAP sobre probabilidades)
    # ==========================================================
    elif "svc" in model_cls:

        background = X_background_proc[:50]

        pos_index = list(estimator.classes_).index(positive_class)

        def model_proba_pos(X):
            return estimator.predict_proba(X)[:, pos_index]

        explainer = shap.KernelExplainer(model_proba_pos, background)

        shap_vals = explainer.shap_values(
            X_shap,
            nsamples=100
        )

        if isinstance(shap_vals, list):
            shap_vals = shap_vals[0]

    # ==========================================================
    # 5. LOGISTIC REGRESSION
    # ==========================================================
    elif "logisticregression" in model_cls:

        explainer = shap.LinearExplainer(
            estimator,
            X_background_proc,
            feature_perturbation="interventional"
        )

        shap_values = explainer(X_shap)

        if shap_values.values.ndim == 3:
            class_index = list(estimator.classes_).index(positive_class)
            shap_vals = shap_values.values[:, :, class_index]
        else:
            shap_vals = shap_values.values

    else:
        raise ValueError(f"Modelo no soportado para SHAP: {estimator.__class__.__name__}")

    # ==========================================================
    # 6. Importancia media absoluta
    # ==========================================================
    mean_abs_shap = np.abs(shap_vals).mean(axis=0)

    importance_df = pd.DataFrame({
        "Descriptor": feature_names,
        "Mean_Abs_SHAP": mean_abs_shap
    }).sort_values("Mean_Abs_SHAP", ascending=False)

    top_df = importance_df.head(top_n)
    top_idx = top_df.index.to_numpy()

    # ==========================================================
    # 7. Plot resumen (solo top features)
    # ==========================================================
    shap.summary_plot(
        shap_vals[:, top_idx],
        X_shap[:, top_idx],
        feature_names=top_df["Descriptor"].values,
        show=True
    )

    shap_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/shap_{target}_{model_name}.png"
    shapcsv_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/shap_importance_{target}_{model_name}.csv"

    plt.savefig(shap_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Gráfico SHAP guardado en: {shap_path}")

    top_df.to_csv(shapcsv_path, index=False)
    print(f"Reporte de importancias guardado en: {shapcsv_path}")

In [ ]:
# ============================================
# 3. Background (submuestreo estable desde TRAIN)
# ============================================
rng = np.random.default_rng(42)

bg_size = min(100, X_train_preprocessed.shape[0])
bg_idx = rng.choice(X_train_preprocessed.shape[0], bg_size, replace=False)

X_background_proc = X_train_preprocessed[bg_idx]

# ============================================
# 5. SHAP Execution (solo si NO stacking)
# ============================================

if use_stacking:
    print("SHAP omitido: modelo stacking no se interpreta directamente sobre descriptores moleculares.")

else:
    print(f"Ejecutando SHAP en train para {model_name} - {target}")

    process_shap_top20(
        estimator=final_model,
        model_name=model_name,
        X_proc=X_train_preprocessed,
        X_background_proc=X_background_proc,
        feature_names=selected_features,
        target=target,
        positive_class=1,
        max_samples=500
    )

## **11. Y-Scrambling**

In [ ]:
scores_scramble = []

print("Iniciando Y-scrambling (control de aleatoriedad)...")

for i in tqdm(range(20), desc="Y-Scrambling"):

    # 1. Permutar etiquetas
    y_scrambled = np.random.permutation(y_train_enc)

    # 2. Clonar el modelo original (NO tocar final_model)
    scramble_model = clone(final_model)

    # 3. Entrenar con etiquetas aleatorias
    scramble_model.fit(X_train_preprocessed, y_scrambled)

    # 4. Evaluación en test externo
    y_pred_scramble = scramble_model.predict(X_test_preprocessed)

    # 5. Métrica
    mcc = matthews_corrcoef(y_test_enc, y_pred_scramble)
    scores_scramble.append(mcc)

# Resultados
yscramble_results_df = pd.DataFrame({
    "Iteration": range(1, len(scores_scramble) + 1),
    "MCC": scores_scramble
})

# Guardar
yscramble_path = f"{BASE_DIR}/outputs_{confidence}/{target}/reports/yscramble_{target}_{model_type}.csv"
yscramble_results_df.to_csv(yscramble_path, index=False)

print(f"Resultados Y-scrambling guardados en: {yscramble_path}")
display(yscramble_results_df)


In [ ]:
# Resultados
mean_yscramble_mcc = yscramble_results_df['MCC'].mean()
std_yscramble_mcc = yscramble_results_df['MCC'].std()
real_mcc = results['metrics_df']['MCC'].iloc[0]

# Gráfica
plt.figure(figsize=(6, 5))
sns.histplot(yscramble_results_df['MCC'], color='skyblue', label='Y-Scrambled MCCs')
plt.axvline(0.0, linestyle=":", linewidth=1.5, label="Random performance (MCC = 0)")
plt.axvline(mean_yscramble_mcc, color='orange', linestyle='--', label=f'Mean Y-Scrambled MCC ({mean_yscramble_mcc:.4f})')
plt.axvline(real_mcc, color='red', linestyle='-', label=f'Real MCC ({real_mcc:.4f})')

plt.title(f'Real MCC vs. Y-Scrambled MCCs ({target} - {model_name})'), plt.xlabel('Matthews Correlation Coefficient (MCC)'), plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)

# Guardado
if model_type == "stacking":
    plot_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/yscramble_{target}_stacking.png"
else:
    plot_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/yscramble_{target}_{model_name}.png"

plt.savefig(plot_path, dpi=300)
plt.show()
print(f"Gráfico de comparación de Y-Scrambling guardado en: {plot_path}")

## **12. Dominio de Aplicabilidad (AD)**

In [ ]:
# AD ACTUALIZADO
# ===============================
# Leverage (descriptor space AD)
# ===============================
n_train, p = X_train_preprocessed.shape

epsilon = 1e-6
XtX_inv = np.linalg.pinv(
    X_train_preprocessed.T @ X_train_preprocessed + epsilon * np.eye(p)
)

# Leverage TRAIN y TEST
leverage_train = np.sum(
    X_train_preprocessed @ XtX_inv * X_train_preprocessed,
    axis=1
)

leverage_test = np.sum(
    X_test_preprocessed @ XtX_inv * X_test_preprocessed,
    axis=1
)

# Umbral AD
h_star = 3 * (p + 1) / n_train

# ===============================
# Porcentaje fuera del AD
# ===============================
outside_ad = leverage_test > h_star
pct_outside = outside_ad.mean() * 100

print(f"Compuestos fuera del AD: {pct_outside:.1f}%")

# ===============================
# Williams plot (clasificación)
# ===============================
plt.figure(figsize=(8, 6))

active_mask = y_test_enc == 1
inactive_mask = y_test_enc == 0

plt.scatter(
    leverage_test[active_mask],
    results['y_proba'][active_mask],
    alpha=0.6,
    label='Active'
)

plt.scatter(
    leverage_test[inactive_mask],
    results['y_proba'][inactive_mask],
    alpha=0.4,
    label='Inactive'
)

plt.axhline(0.5, linestyle='--', linewidth=1, label='Decision threshold')
plt.axvline(h_star, linestyle='--', linewidth=1.5, label='AD threshold')
# Añadir el porcentaje de compuestos fuera del AD al gráfico
plt.text(0.98, 0.02, f'{pct_outside:.1f}% outside AD', transform=plt.gca().transAxes,
    fontsize=10, color='black', horizontalalignment='right', verticalalignment='bottom'
)

plt.xlabel('Leverage (descriptor space)')
plt.ylabel('Predicted probability (Active)')
plt.title(f'Applicability Domain ({target} – {model_type})')
plt.legend()
plt.grid(alpha=0.3)

plot_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/ad_{target}_{model_type}.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"Diagrama de AD guardado en: {plot_path}")

In [ ]:
# APLICACIÓN DEL MODELO

def plot_qsar_mode(metrics_df):
    recall = metrics_df.loc[0, "Recall"]
    specificity = metrics_df.loc[0, "Specificity"]

    plt.figure(figsize=(6, 6))

    # Zonas
    plt.axvspan(0, 0.60, ymin=0.85, ymax=1.0, alpha=0.2)
    plt.axvspan(0.60, 0.75, ymin=0.70, ymax=0.85, alpha=0.2)
    plt.axvspan(0.75, 1.0, ymin=0.0, ymax=0.70, alpha=0.2)

    # Punto del modelo
    plt.scatter(specificity, recall, s=100)
    plt.text(specificity + 0.01, recall, f"  {model_type}", fontsize=10)

    plt.xlabel("Specificity")
    plt.ylabel("Recall")
    plt.title("QSAR Toxicology Mode Diagnosis")

    plt.text(0.15, 0.95, "Screening", fontsize=11, weight="bold")
    plt.text(0.62, 0.78, "Balanced", fontsize=11, weight="bold")
    plt.text(0.82, 0.25, "Regulatory", fontsize=11, weight="bold")
    plt.annotate("Threshold = 0.5", (specificity, recall), textcoords="offset points", xytext=(10, -15))
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(True)

    mode_path = f"{BASE_DIR}/outputs_{confidence}/{target}/plots/qsar_mode_{target}_{model_name}.png"
    plt.savefig(mode_path)
    print(f"Gráfico de modo guardado en: {mode_path}")
    plt.show()

    return None

plot_qsar_mode(results['metrics_df'])


## **13. Predicción externa**

In [ ]:
def mol_to_base64_image(mol, width=200, height=200):
    if mol is None:
        return None
    try:
        img = Draw.MolToImage(mol, size=(width, height))
        buffered = io.BytesIO()
        img.save(buffered, format="PNG")
        return base64.b64encode(buffered.getvalue()).decode("utf-8")
    except Exception as e:
        print(f"Error generating image for molecule: {e}")
        return None

def predict_from_smiles(smiles_list):

    processed_data = []
    original_smiles = []
    molecule_images = []
    valid_mols = []
    invalid_smiles_list = []

    for s in smiles_list:

        mol = Chem.MolFromSmiles(s)

        if mol is None:
            invalid_smiles_list.append(s)
            continue

        try:
            lfc = rdMolStandardize.LargestFragmentChooser()
            normalizer = rdMolStandardize.Normalizer()
            reionizer = rdMolStandardize.Reionizer()
            uncharger = rdMolStandardize.Uncharger()

            mol = lfc.choose(mol)
            mol = normalizer.normalize(mol)
            mol = reionizer.reionize(mol)
            mol = uncharger.uncharge(mol)

            smiles_std = Chem.MolToSmiles(mol)
            descriptors = calcular_descriptores(smiles_std)

            if all(d is not None and not np.isnan(d) for d in descriptors):
                processed_data.append(descriptors)
                original_smiles.append(s)
                valid_mols.append(mol)
                molecule_images.append(mol_to_base64_image(mol))
            else:
                invalid_smiles_list.append(s)

        except Exception:
            invalid_smiles_list.append(s)

    predictions_df = pd.DataFrame()

    if processed_data:

        # ===============================
        # Construcción DataFrame
        # ===============================
        raw_X_ext_df = pd.DataFrame(
            processed_data,
            columns=descriptor_names_full
        )

        raw_X_ext = raw_X_ext_df[selected_features].values

        # ===============================
        # Preprocesamiento
        # ===============================
        X_ext_preprocessed = trained_preprocessor.transform(raw_X_ext)

        # ===============================
        # Predicción
        # ===============================

        # Probabilidades para todas las clases
        y_proba_all = final_model.predict_proba(X_ext_preprocessed)

        # Obtener índice real de la clase 1 (Active)
        classes = list(final_model.classes_)
        if 1 not in classes:
            raise ValueError("El modelo no contiene la clase 1 (Active).")

        active_index = classes.index(1)

        # Probabilidad correcta de ACTIVE
        y_proba_active = y_proba_all[:, active_index]

        # Predicción binaria usando threshold estándar 0.5
        y_pred_labels = np.where(
            y_proba_active >= 0.5,
            "Active",
            "Inactive"
        )
        # ===============================
        # Dominio de Aplicabilidad
        # ===============================
        leverage_ext = np.sum(
            X_ext_preprocessed @ XtX_inv * X_ext_preprocessed,
            axis=1
        )

        ad_flag = np.where(
            leverage_ext <= h_star,
            "Inside AD",
            "Outside AD"
        )

        predictions_df = pd.DataFrame({
            "SMILES": original_smiles,
            "Activity_Prediction": y_pred_labels,
            "Probability_Active": y_proba_active,
            "Leverage": leverage_ext,
            "AD_Flag": ad_flag,
            "Molecule_Image": molecule_images
        })

    invalid_df = pd.DataFrame({
        "Invalid_SMILES": invalid_smiles_list,
        "Reason": "Invalid SMILES or descriptor calculation failed"
    }) if invalid_smiles_list else pd.DataFrame(
        columns=["Invalid_SMILES", "Reason"]
    )

    return predictions_df, invalid_df

# Helper function to display DataFrame with images
def display_df_with_images(df):
    if 'Molecule_Image' in df.columns:
        html_output = df.to_html(escape=False, formatters=dict(Molecule_Image=lambda img: f'<img src="data:image/png;base64,{img}">'))
        display(HTML(html_output))
    else:
        display(df)

In [ ]:
def predict_and_rank_actives(smiles_list, top_n=None):

    predictions_df, invalid_df = predict_from_smiles(smiles_list)

    if predictions_df.empty:
        return predictions_df, invalid_df

    # ===============================
    # Filtrar solo activos dentro del AD
    # ===============================
    ranked_df = predictions_df[
        (predictions_df["Activity_Prediction"] == "Active") &
        (predictions_df["AD_Flag"] == "Inside AD")
    ].copy()

    # ===============================
    # Ordenar por probabilidad de Active
    # ===============================
    ranked_df = ranked_df.sort_values(
        by="Probability_Active",
        ascending=False
    ).reset_index(drop=True)

    # ===============================
    # Ranking numérico
    # ===============================
    ranked_df.insert(0, "Rank", range(1, len(ranked_df) + 1))

    if top_n is not None:
        ranked_df = ranked_df.head(top_n)

    return ranked_df, invalid_df

In [ ]:
# Predicción de ejemplo
smiles_test = ['Clc1cc2c(cc1Cl)Oc1cc(Cl)c(Cl)cc1O2', 'O=C(O)c1cccc(-c2noc(-c3ccccc3F)n2)c1', 'Cc1nc(-c2ccc(OCC(C)C)c(C#N)c2)sc1C(=O)O', 'COc1c2ccoc2cc2oc(=O)ccc12', 'c1ccc2cc3c(cc2c1)-c1cccc2cccc-3c12']

predictions, invalid_smiles = predict_from_smiles(smiles_test)

display_df_with_images(predictions)

ranked_df, invalid_df = predict_and_rank_actives(smiles_test, top_n=10)

display_df_with_images(ranked_df)

In [ ]:
final_model.classes_

In [ ]:
proba = final_model.predict_proba(X_test_preprocessed)

In [ ]:
proba